# Preprocessing of the "Raw" Data

In [18]:
import pandas as pd
import random
import string
import re
random.seed(42)

In [38]:
raw_df = pd.read_csv("../AP1/issue_df.csv")
print(raw_df.head())
# print(len(raw_df[raw_df["Party"] == "R"])/len(raw_df))
prop_republican = len(raw_df[raw_df["Party"] == "R"])/len(raw_df)
prop_dem = 1 - prop_republican

               Name    State District Party  \
0  Aderholt, Robert  Alabama      4th     R   
1  Aderholt, Robert  Alabama      4th     R   
2  Aderholt, Robert  Alabama      4th     R   
3  Aderholt, Robert  Alabama      4th     R   
4  Aderholt, Robert  Alabama      4th     R   

                                             Content  \
0  I believe the majority of Alabamians support s...   
1  "Knowledge is power." Thomas Jefferson, Januar...   
2  Energy becomes a more important topic each and...   
3  "Congress shall make no law respecting an esta...   
4  The Founding Fathers wisely included the 2nd A...   

                                          Issue Link  
0          https://aderholt.house.gov/issues/economy  
1        https://aderholt.house.gov/issues/education  
2           https://aderholt.house.gov/issues/energy  
3  https://aderholt.house.gov/issues/faith-and-re...  
4       https://aderholt.house.gov/issues/gun-rights  


In [39]:
name_to_anonymized = {}
clean_df = raw_df.copy()
for name in set(raw_df["Name"]):
    clean_df.loc[clean_df["Name"] == name, "Name"] = "".join(
        random.choices(string.ascii_uppercase + string.digits, k=10)
    )
clean_df.drop(columns=["State", "District", "Issue Link"], inplace=True)
clean_df.head()

,Name,Party,Content
0,NZRF5A7QY8,R,I believe the majority of Alabamians support s...
1,NZRF5A7QY8,R,"""Knowledge is power."" Thomas Jefferson, Januar..."
2,NZRF5A7QY8,R,Energy becomes a more important topic each and...
3,NZRF5A7QY8,R,"""Congress shall make no law respecting an esta..."
4,NZRF5A7QY8,R,The Founding Fathers wisely included the 2nd A...


In [40]:
old_length = len(clean_df)
for name, party, content in zip(clean_df["Name"], clean_df["Party"], clean_df["Content"]):
    sentence_content = content.split(".")
    if len(sentence_content) >= 3:
        for i in range(len(sentence_content) - 2):
            three_sentence = sentence_content[i] + "." + sentence_content[i + 1] + "." + sentence_content[i + 2]
            clean_df.loc[len(clean_df)] = [name, party, three_sentence]
        else:
            clean_df.loc[len(clean_df)] = [name, party, None]
clean_df = clean_df.dropna()
clean_df = clean_df.iloc[old_length:]
clean_df.reset_index(drop=True, inplace=True)

clean_df

,Name,Party,Content
0,NZRF5A7QY8,R,I believe the majority of Alabamians support s...
1,NZRF5A7QY8,R,"We need to protect taxpayers, ensure our seni..."
2,NZRF5A7QY8,R,We have seen continually weak growth in the ec...
3,NZRF5A7QY8,R,"The time for action is now. In May 2012, I vo..."
4,NZRF5A7QY8,R,"In May 2012, I voted to replace the sequester..."
...,...,...,...
21368,O091C795RL,R,has delegated authority to unelected bureaucr...
21369,O091C795RL,R,\nAdministrative agencies have the power to w...
21370,O091C795RL,R,How this translates in Wyoming is the EPA’s a...
21371,O091C795RL,R,"As an attorney, I fought to return control of ..."


In [43]:
republican_samples = clean_df[clean_df["Party"] == "R"].sample(n=round(50*prop_republican), random_state=42)
democratic_samples = clean_df[clean_df["Party"] == "D"].sample(n=50 - round(50*prop_republican), random_state=42)
ap3_df = pd.concat([republican_samples, democratic_samples]).reset_index(drop=True)
ap3_df = ap3_df.sample(frac=1, random_state=42).reset_index(drop=True)
ap3_df.drop(columns=["Party"], inplace=True)
ap3_df["Annotator_1"] = None
ap3_df["Annotator_2"] = None
ap3_df["Annotator_3"] = None
ap3_df["Annotator_4"] = None
ap3_df.head()

,Name,Content,Annotator_1,Annotator_2,Annotator_3,Annotator_4
0,OLNNTXRGN7,Permanently altering and disabling the physic...,None,None,None,None
1,NTBZ3M55IZ,NO CORRUPTION Act: Politicians should face re...,None,None,None,None
2,HQR6F7QNEZ,\nCareer and Technical Education and Workforce...,None,None,None,None
3,U8EL5015OO,Congresswoman Garcia works diligently to ens...,None,None,None,None
4,G6W3STT1EQ,Disposable Income is projected to increase b...,None,None,None,None


In [44]:
ap3_df.to_csv("ap3_df.tsv", sep="\t", index=False)